# Annular Ablation Edge Detection

Automated edge detection and ellipse fitting to match Fiji CSV ground truth (`BX`, `BY`, `Major`, `Minor`, `Angle`).

1. Split **discs** into train / test (frames from the same disc stay together)
2. Tune edge-detection parameters on the train set
3. Fit a Ridge correction model from raw detections → Fiji labels
4. Evaluate on held-out test discs

In [ ]:
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np

from ablation_edge.data import AblationDataset, load_gray_image, split_discs, list_disc_ids, save_split
from ablation_edge.detector import AblationEdgeDetector
from ablation_edge.evaluate import evaluate_predictions, print_metrics, metrics_by_disc
from ablation_edge.train import tune_detector, run_evaluation

DATA_DIR = Path("Data")
SPLIT_PATH = Path("splits/train_test_split.json")

In [ ]:
disc_ids = list_disc_ids(DATA_DIR)
train_ids, test_ids = split_discs(disc_ids, test_fraction=0.2, seed=42)
SPLIT_PATH.parent.mkdir(exist_ok=True)
save_split(train_ids, test_ids, SPLIT_PATH)

print(f"Total discs: {len(disc_ids)}")
print(f"Train discs: {len(train_ids)}")
print(f"Test discs:  {len(test_ids)}")
print("Test disc IDs:", test_ids[:5], "...")

In [ ]:
best_params = tune_detector(DATA_DIR, train_ids, max_samples=300)
detector = AblationEdgeDetector(**{k: v for k, v in best_params.items() if k != "tuning_score"})
print("Tuned parameters:")
for key, value in detector.params.items():
    print(f"  {key}: {value}")
print(f"Tuning score: {best_params.get('tuning_score'):.2f}")

In [ ]:
train_images, train_gts = [], []
for sample in AblationDataset(DATA_DIR, train_ids):
    image = load_gray_image(sample.image_path)
    raw = detector._best_raw_detection(image)
    if raw is None:
        continue
    train_images.append(image)
    train_gts.append(sample.ground_truth)

detector.fit_correction(train_images, train_gts)
print(f"Correction model trained on {len(train_images)} frames")

In [ ]:
train_eval, _ = run_evaluation(DATA_DIR, train_ids, detector, label="Train set")
test_eval, _ = run_evaluation(DATA_DIR, test_ids, detector, label="Test set")

metrics_by_disc(test_eval).head(10)

In [ ]:
# Visualize one test-frame: detected edge ellipse vs Fiji ground truth

sample = next(iter(AblationDataset(DATA_DIR, test_ids)))
image = load_gray_image(sample.image_path)
result = detector.detect(image)
gt = sample.ground_truth

fig, ax = plt.subplots(figsize=(7, 7))
ax.imshow(image, cmap="gray", vmin=0, vmax=80)

for label, params, color in [
    ("Fiji GT", gt, "lime"),
    ("Detected", result.as_dict() if result else None, "cyan"),
]:
    if params is None:
        continue
    center = (int(params["BX"]), int(params["BY"]))
    axes = (max(int(params["Major"] / 2), 1), max(int(params["Minor"] / 2), 1))
    cv2.ellipse(
        overlay := np.zeros_like(image),
        center,
        axes,
        params["Angle"],
        0,
        360,
        255,
        2,
    )
    patch = np.ma.masked_where(overlay == 0, overlay)
    ax.imshow(patch, cmap="autumn" if color == "lime" else "winter", alpha=0.35)
    ax.plot(params["BX"], params["BY"], "+", color=color, ms=12, label=label)

ax.set_title(f"{sample.disc_id} frame {sample.frame:04d}")
ax.legend()
ax.axis("off")
plt.show()